In [1]:
# ============================================================
# ERAPAVE-LIKE V2.3
# Direct regularized calibration of ERAPave D0 and D30
# ============================================================

from pathlib import Path
import importlib.util

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneOut, GridSearchCV
from sklearn.metrics import mean_absolute_percentage_error

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ============================================================
# LOAD RAW MLET SOLVER
# ============================================================

HERE = Path.cwd()

RAW_SOLVER = HERE / "ERAPave_MLET_V3_RAW_Benchmark.py"

if not RAW_SOLVER.exists():

    # Try the parent directory
    candidates = [
        HERE / "ERAPave_MLET_V3_RAW_Benchmark.py",
        HERE.parent / "ERAPave_MLET_V3_RAW_Benchmark.py",
        Path("/mnt/data/ERAPave_MLET_V3_RAW_Benchmark.py")
    ]

    found = None

    for p in candidates:
        if p.exists():
            found = p
            break

    if found is None:
        raise FileNotFoundError(
            "Could not find ERAPave_MLET_V3_RAW_Benchmark.py\n"
            "Put the file in the same folder as this notebook."
        )

    RAW_SOLVER = found

print("Raw solver:")
print(RAW_SOLVER)

spec = importlib.util.spec_from_file_location(
    "raw_mlet",
    RAW_SOLVER
)

raw_mlet = importlib.util.module_from_spec(spec)
spec.loader.exec_module(raw_mlet)

print("Raw MLET solver loaded successfully.")

Raw solver:
d:\THESIS\VTI\notebooks\ERAPave_MLET_V3_RAW_Benchmark.py
Raw MLET solver loaded successfully.


In [ ]:
# ============================================================
# ROW 23 STRUCTURE
# ============================================================

# 2. CURRENT ROW 24 STRUCTURE

THICKNESS_CM = np.array(
    [13.4246, 42.3460, 87.1116],
    dtype=float
)

POISSON = np.array([
    0.35,
    0.35,
    0.35,
    0.40
], dtype=float)

PRESSURE_KPA = 700.0
LOAD_KN = 50.0

# IMPORTANT:
# D0 = x = 0 cm
# D30 = x = 30 cm
LOCATIONS_CM = np.array([
    0,
    20,
    30,
    60,
    90,
    120,
    150
], dtype=float)

# Contact radius
RADIUS_CM = np.sqrt(
    LOAD_KN / (np.pi * PRESSURE_KPA)
) * 100.0

print("Row 23 structure")
print("----------------")
print("Thickness =", THICKNESS_CM, "cm")
print("Poisson   =", POISSON)
print("Pressure  =", PRESSURE_KPA, "kPa")
print("Load      =", LOAD_KN, "kN")
print("Radius    =", round(RADIUS_CM, 4), "cm")
print("Locations =", LOCATIONS_CM, "cm")

Row 23 structure
----------------
Thickness = [13.989 37.085 88.263] cm
Poisson   = [0.35 0.35 0.35 0.4 ]
Pressure  = 700.0 kPa
Load      = 50.0 kN
Radius    = 15.0786 cm
Locations = [  0.  20.  30.  60.  90. 120. 150.] cm


In [4]:
# ============================================================
# 10 ACTUAL ERAPAVE CALIBRATION POINTS
#
# Columns:
# E1, E2, E3, E4, ERAPave D0, ERAPave D30
#
# D0  = displacement at x = 0 cm
# D30 = displacement at x = 30 cm
# ============================================================

DATA = np.array([

    # Original 8 points
    [840, 195,  95, 20, 584.6, 359.5],
    [700, 160,  90, 20, 653.5, 383.6],
    [650, 180,  90, 20, 645.6, 375.3],
    [580, 180,  90, 20, 666.3, 377.2],
    [550, 200,  90, 20, 657.5, 370.2],
    [500, 220, 120, 20, 628.4, 337.1],
    [450, 180, 120, 20, 685.3, 333.4],
    [450, 250, 160, 20, 601.0, 306.1],

    # Actual ERAPave point:
    # 450 / 200 / 130 / 20
    [450, 200, 130, 20, 657.7, 337.2],

    # Actual ERAPave point:
    # 450 / 180 / 130 / 20
    [450, 180, 130, 20, 676.9, 344.4],

    # NEW actual ERAPave point:
    # 450 / 180 / 140 / 20
    [450, 180, 140, 20, 669.4, 337.7],

], dtype=float)


print("Number of ERAPave calibration points =", len(DATA))

Number of ERAPave calibration points = 11


In [5]:
# ============================================================
# DATAFRAME
# ============================================================

columns = [
    "E1",
    "E2",
    "E3",
    "E4",
    "ERAPave_D0",
    "ERAPave_D30"
]

df = pd.DataFrame(DATA, columns=columns)

df["ERAPave_SCI300"] = (
    df["ERAPave_D0"] -
    df["ERAPave_D30"]
)

df

,E1,E2,E3,E4,ERAPave_D0,ERAPave_D30,ERAPave_SCI300
0,840.0,195.0,95.0,20.0,584.6,359.5,225.1
1,700.0,160.0,90.0,20.0,653.5,383.6,269.9
2,650.0,180.0,90.0,20.0,645.6,375.3,270.3
3,580.0,180.0,90.0,20.0,666.3,377.2,289.1
4,550.0,200.0,90.0,20.0,657.5,370.2,287.3
5,500.0,220.0,120.0,20.0,628.4,337.1,291.3
6,450.0,180.0,120.0,20.0,685.3,333.4,351.9
7,450.0,250.0,160.0,20.0,601.0,306.1,294.9
8,450.0,200.0,130.0,20.0,657.7,337.2,320.5
9,450.0,180.0,130.0,20.0,676.9,344.4,332.5


In [6]:
# ============================================================
# RAW MLET FORWARD CALCULATION
# ============================================================

def raw_mlet_displacements(E):

    E = np.asarray(E, dtype=float)

    raw = raw_mlet.erapave_mlet(
        q_kpa=PRESSURE_KPA,
        radius_cm=RADIUS_CM,
        r_cm=LOCATIONS_CM,
        z_cm=np.zeros_like(LOCATIONS_CM),
        thickness_cm=THICKNESS_CM,
        modulus_mpa=E,
        poisson=POISSON,
        iteration=80
    )

    raw = np.asarray(raw, dtype=float).ravel()

    # Locations:
    # index 0 -> x=0 cm
    # index 1 -> x=20 cm
    # index 2 -> x=30 cm

    D0_raw = raw[0]
    D30_raw = raw[2]

    return D0_raw, D30_raw

In [7]:
# ============================================================
# CALCULATE RAW MLET FOR ALL ERAPAVE POINTS
# ============================================================

raw_results = []

for _, row in df.iterrows():

    E = [
        row["E1"],
        row["E2"],
        row["E3"],
        row["E4"]
    ]

    d0_raw, d30_raw = raw_mlet_displacements(E)

    raw_results.append([
        d0_raw,
        d30_raw
    ])


raw_results = np.asarray(raw_results)

df["Raw_D0"] = raw_results[:, 0]
df["Raw_D30"] = raw_results[:, 1]

df["Raw_SCI300"] = (
    df["Raw_D0"] -
    df["Raw_D30"]
)

df

,E1,E2,E3,E4,ERAPave_D0,ERAPave_D30,ERAPave_SCI300,Raw_D0,Raw_D30,Raw_SCI300
0,840.0,195.0,95.0,20.0,584.6,359.5,225.1,1048.279660,720.479314,327.800346
1,700.0,160.0,90.0,20.0,653.5,383.6,269.9,1161.681453,769.010321,392.671132
2,650.0,180.0,90.0,20.0,645.6,375.3,270.3,1139.395416,752.649198,386.746218
3,580.0,180.0,90.0,20.0,666.3,377.2,289.1,1165.737988,756.642667,409.095320
4,550.0,200.0,90.0,20.0,657.5,370.2,287.3,1143.239100,742.681928,400.557172
5,500.0,220.0,120.0,20.0,628.4,337.1,291.3,1073.612539,677.101183,396.511356
6,450.0,180.0,120.0,20.0,685.3,333.4,351.9,1164.489000,707.330315,457.158686
7,450.0,250.0,160.0,20.0,601.0,306.1,294.9,1005.589371,615.646161,389.943210
8,450.0,200.0,130.0,20.0,657.7,337.2,320.5,1112.409248,677.930084,434.479164
9,450.0,180.0,130.0,20.0,676.9,344.4,332.5,1147.839326,692.412288,455.427037


In [8]:
# ============================================================
# RAW MLET CHECK
# ============================================================

print("=" * 100)
print("RAW MLET VS ERAPAVE")
print("=" * 100)

print(
    f"{'Run':>4} "
    f"{'E1':>6} "
    f"{'E2':>6} "
    f"{'E3':>6} "
    f"{'E4':>6} "
    f"{'ERA D0':>10} "
    f"{'Raw D0':>10} "
    f"{'ERA D30':>10} "
    f"{'Raw D30':>10}"
)

print("-" * 100)

for i, row in df.iterrows():

    print(
        f"{i:4d} "
        f"{row.E1:6.0f} "
        f"{row.E2:6.0f} "
        f"{row.E3:6.0f} "
        f"{row.E4:6.0f} "
        f"{row.ERAPave_D0:10.2f} "
        f"{row.Raw_D0:10.2f} "
        f"{row.ERAPave_D30:10.2f} "
        f"{row.Raw_D30:10.2f}"
    )

RAW MLET VS ERAPAVE
 Run     E1     E2     E3     E4     ERA D0     Raw D0    ERA D30    Raw D30
----------------------------------------------------------------------------------------------------
   0    840    195     95     20     584.60    1048.28     359.50     720.48
   1    700    160     90     20     653.50    1161.68     383.60     769.01
   2    650    180     90     20     645.60    1139.40     375.30     752.65
   3    580    180     90     20     666.30    1165.74     377.20     756.64
   4    550    200     90     20     657.50    1143.24     370.20     742.68
   5    500    220    120     20     628.40    1073.61     337.10     677.10
   6    450    180    120     20     685.30    1164.49     333.40     707.33
   7    450    250    160     20     601.00    1005.59     306.10     615.65
   8    450    200    130     20     657.70    1112.41     337.20     677.93
   9    450    180    130     20     676.90    1147.84     344.40     692.41
  10    450    180    140     20

In [9]:
# ============================================================
# V2.3 FEATURES
# ============================================================

def make_features(dataframe):

    X = np.column_stack([

        # Raw MLET responses
        np.log(dataframe["Raw_D0"].values),
        np.log(dataframe["Raw_D30"].values),

        # Layer moduli
        np.log(dataframe["E1"].values),
        np.log(dataframe["E2"].values),
        np.log(dataframe["E3"].values),
        np.log(dataframe["E4"].values),

    ])

    return X


X = make_features(df)

feature_names = [
    "log(Raw_D0)",
    "log(Raw_D30)",
    "log(E1)",
    "log(E2)",
    "log(E3)",
    "log(E4)"
]

print("V2.3 features:")
for i, name in enumerate(feature_names):
    print(i, name)

print()
print("Feature matrix shape =", X.shape)

V2.3 features:
0 log(Raw_D0)
1 log(Raw_D30)
2 log(E1)
3 log(E2)
4 log(E3)
5 log(E4)

Feature matrix shape = (11, 6)


In [10]:
# ============================================================
# TARGETS
# ============================================================

y_D0 = np.log(df["ERAPave_D0"].values)

y_D30 = np.log(df["ERAPave_D30"].values)

print("D0 target shape :", y_D0.shape)
print("D30 target shape:", y_D30.shape)

D0 target shape : (11,)
D30 target shape: (11,)


In [14]:
# ============================================================
# V2.3 RIDGE MODEL
# ============================================================

def make_model(alpha):

    return Pipeline([
        (
            "scaler",
            StandardScaler()
        ),

        (
            "ridge",
            Ridge(alpha=alpha)
        )
    ])


# Initial regularization
ALPHA = 1.0

model_D0 = make_model(ALPHA)
model_D30 = make_model(ALPHA)

model_D0.fit(X, y_D0)
model_D30.fit(X, y_D30)

print("V2.3 models trained.")

V2.3 models trained.


In [15]:
# ============================================================
# TRAINING PREDICTIONS
# ============================================================

pred_log_D0 = model_D0.predict(X)
pred_log_D30 = model_D30.predict(X)

pred_D0 = np.exp(pred_log_D0)
pred_D30 = np.exp(pred_log_D30)

pred_SCI = pred_D0 - pred_D30

true_D0 = df["ERAPave_D0"].values
true_D30 = df["ERAPave_D30"].values
true_SCI = df["ERAPave_SCI300"].values


df["Pred_D0"] = pred_D0
df["Pred_D30"] = pred_D30
df["Pred_SCI300"] = pred_SCI

df["D0_Error_pct"] = (
    100.0 *
    (pred_D0 - true_D0) /
    true_D0
)

df["D30_Error_pct"] = (
    100.0 *
    (pred_D30 - true_D30) /
    true_D30
)

df["SCI_Error_pct"] = (
    100.0 *
    (pred_SCI - true_SCI) /
    true_SCI
)

In [16]:
# ============================================================
# TRAINING PREDICTIONS
# ============================================================

pred_log_D0 = model_D0.predict(X)
pred_log_D30 = model_D30.predict(X)

pred_D0 = np.exp(pred_log_D0)
pred_D30 = np.exp(pred_log_D30)

pred_SCI = pred_D0 - pred_D30

true_D0 = df["ERAPave_D0"].values
true_D30 = df["ERAPave_D30"].values
true_SCI = df["ERAPave_SCI300"].values


df["Pred_D0"] = pred_D0
df["Pred_D30"] = pred_D30
df["Pred_SCI300"] = pred_SCI

df["D0_Error_pct"] = (
    100.0 *
    (pred_D0 - true_D0) /
    true_D0
)

df["D30_Error_pct"] = (
    100.0 *
    (pred_D30 - true_D30) /
    true_D30
)

df["SCI_Error_pct"] = (
    100.0 *
    (pred_SCI - true_SCI) /
    true_SCI
)

In [17]:
# ============================================================
# TRAINING PERFORMANCE
# ============================================================

d0_mape = np.mean(
    np.abs(df["D0_Error_pct"])
)

d30_mape = np.mean(
    np.abs(df["D30_Error_pct"])
)

sci_mape = np.mean(
    np.abs(df["SCI_Error_pct"])
)

print("=" * 70)
print("V2.3 TRAINING PERFORMANCE")
print("=" * 70)

print(f"D0     MAPE = {d0_mape:.2f}%")
print(f"D30    MAPE = {d30_mape:.2f}%")
print(f"SCI300 MAPE = {sci_mape:.2f}%")

V2.3 TRAINING PERFORMANCE
D0     MAPE = 0.29%
D30    MAPE = 1.04%
SCI300 MAPE = 1.01%


In [18]:
# ============================================================
# TRAINING PREDICTIONS
# ============================================================

print("=" * 125)
print("V2.3 TRAINING PREDICTIONS")
print("=" * 125)

print(
    f"{'Run':>4} "
    f"{'E1':>6} "
    f"{'E2':>6} "
    f"{'E3':>6} "
    f"{'E4':>6} "
    f"{'ERA D0':>9} "
    f"{'Pred D0':>9} "
    f"{'ERA D30':>9} "
    f"{'Pred D30':>9} "
    f"{'D0 err%':>9} "
    f"{'D30 err%':>10} "
    f"{'SCI err%':>10}"
)

print("-" * 125)

for i, row in df.iterrows():

    print(
        f"{i:4d} "
        f"{row.E1:6.0f} "
        f"{row.E2:6.0f} "
        f"{row.E3:6.0f} "
        f"{row.E4:6.0f} "
        f"{row.ERAPave_D0:9.1f} "
        f"{row.Pred_D0:9.1f} "
        f"{row.ERAPave_D30:9.1f} "
        f"{row.Pred_D30:9.1f} "
        f"{row.D0_Error_pct:9.2f} "
        f"{row.D30_Error_pct:10.2f} "
        f"{row.SCI_Error_pct:10.2f}"
    )

V2.3 TRAINING PREDICTIONS
 Run     E1     E2     E3     E4    ERA D0   Pred D0   ERA D30  Pred D30   D0 err%   D30 err%   SCI err%
-----------------------------------------------------------------------------------------------------------------------------
   0    840    195     95     20     584.6     589.8     359.5     363.4      0.89       1.09       0.58
   1    700    160     90     20     653.5     654.9     383.6     381.1      0.21      -0.64       1.42
   2    650    180     90     20     645.6     646.1     375.3     373.4      0.07      -0.50       0.86
   3    580    180     90     20     666.3     664.5     377.2     373.5     -0.27      -0.99       0.67
   4    550    200     90     20     657.5     654.6     370.2     366.9     -0.44      -0.90       0.15
   5    500    220    120     20     628.4     629.2     337.1     336.2      0.12      -0.26       0.57
   6    450    180    120     20     685.3     682.2     333.4     347.9     -0.45       4.35      -4.99
   7    

In [19]:
# ============================================================
# V2.3 LEAVE-ONE-OUT VALIDATION
# ============================================================

loo = LeaveOneOut()

loo_results = []

for train_idx, test_idx in loo.split(X):

    X_train = X[train_idx]
    X_test = X[test_idx]

    y0_train = y_D0[train_idx]
    y30_train = y_D30[train_idx]

    # --------------------------------------------------------
    # Train D0 model
    # --------------------------------------------------------

    model0 = make_model(ALPHA)

    model0.fit(
        X_train,
        y0_train
    )

    # --------------------------------------------------------
    # Train D30 model
    # --------------------------------------------------------

    model30 = make_model(ALPHA)

    model30.fit(
        X_train,
        y30_train
    )

    # --------------------------------------------------------
    # Predict left-out point
    # --------------------------------------------------------

    d0_pred = np.exp(
        model0.predict(X_test)
    )[0]

    d30_pred = np.exp(
        model30.predict(X_test)
    )[0]

    # Actual
    idx = test_idx[0]

    d0_true = true_D0[idx]
    d30_true = true_D30[idx]

    sci_true = d0_true - d30_true

    sci_pred = d0_pred - d30_pred

    # Errors
    d0_err = 100.0 * (
        d0_pred - d0_true
    ) / d0_true

    d30_err = 100.0 * (
        d30_pred - d30_true
    ) / d30_true

    sci_err = 100.0 * (
        sci_pred - sci_true
    ) / sci_true

    loo_results.append([
        idx,
        df.loc[idx, "E1"],
        df.loc[idx, "E2"],
        df.loc[idx, "E3"],
        df.loc[idx, "E4"],
        d0_true,
        d0_pred,
        d30_true,
        d30_pred,
        d0_err,
        d30_err,
        sci_err
    ])


loo_df = pd.DataFrame(
    loo_results,
    columns=[
        "Run",
        "E1",
        "E2",
        "E3",
        "E4",
        "ERA_D0",
        "Pred_D0",
        "ERA_D30",
        "Pred_D30",
        "D0_Error_pct",
        "D30_Error_pct",
        "SCI_Error_pct"
    ]
)

loo_df

,Run,E1,E2,E3,E4,ERA_D0,Pred_D0,ERA_D30,Pred_D30,D0_Error_pct,D30_Error_pct,SCI_Error_pct
0,0,840.0,195.0,95.0,20.0,584.6,595.354741,359.5,372.067629,1.839675,3.495863,-0.805370
1,1,700.0,160.0,90.0,20.0,653.5,655.761984,383.6,379.852259,0.346134,-0.976992,2.226649
2,2,650.0,180.0,90.0,20.0,645.6,646.167812,375.3,372.997796,0.087951,-0.613430,1.061789
3,3,580.0,180.0,90.0,20.0,666.3,663.868650,377.2,372.356391,-0.364903,-1.284096,0.834403
4,4,550.0,200.0,90.0,20.0,657.5,652.369633,370.2,364.709312,-0.780284,-1.483168,0.125416
5,5,500.0,220.0,120.0,20.0,628.4,629.269848,337.1,335.917013,0.138423,-0.350931,0.704715
6,6,450.0,180.0,120.0,20.0,685.3,681.313305,333.4,352.267712,-0.581744,5.659182,-6.494574
7,7,450.0,250.0,160.0,20.0,601.0,603.458130,306.1,307.816042,0.409007,0.560615,0.251641
8,8,450.0,200.0,130.0,20.0,657.7,655.970529,337.2,334.659852,-0.262957,-0.753306,0.252941
9,9,450.0,180.0,130.0,20.0,676.9,674.908536,344.4,340.168935,-0.294204,-1.228532,0.673564


In [20]:
# ============================================================
# DISPLAY LOO
# ============================================================

print("=" * 125)
print("V2.3 LEAVE-ONE-OUT VALIDATION")
print("=" * 125)

print(
    f"{'Run':>4} "
    f"{'E1':>6} "
    f"{'E2':>6} "
    f"{'E3':>6} "
    f"{'E4':>6} "
    f"{'ERA D0':>9} "
    f"{'Pred D0':>9} "
    f"{'ERA D30':>9} "
    f"{'Pred D30':>9} "
    f"{'D0 err%':>9} "
    f"{'D30 err%':>10} "
    f"{'SCI err%':>10}"
)

print("-" * 125)

for _, row in loo_df.iterrows():

    print(
        f"{int(row.Run):4d} "
        f"{row.E1:6.0f} "
        f"{row.E2:6.0f} "
        f"{row.E3:6.0f} "
        f"{row.E4:6.0f} "
        f"{row.ERA_D0:9.1f} "
        f"{row.Pred_D0:9.1f} "
        f"{row.ERA_D30:9.1f} "
        f"{row.Pred_D30:9.1f} "
        f"{row.D0_Error_pct:9.2f} "
        f"{row.D30_Error_pct:10.2f} "
        f"{row.SCI_Error_pct:10.2f}"
    )

V2.3 LEAVE-ONE-OUT VALIDATION
 Run     E1     E2     E3     E4    ERA D0   Pred D0   ERA D30  Pred D30   D0 err%   D30 err%   SCI err%
-----------------------------------------------------------------------------------------------------------------------------
   0    840    195     95     20     584.6     595.4     359.5     372.1      1.84       3.50      -0.81
   1    700    160     90     20     653.5     655.8     383.6     379.9      0.35      -0.98       2.23
   2    650    180     90     20     645.6     646.2     375.3     373.0      0.09      -0.61       1.06
   3    580    180     90     20     666.3     663.9     377.2     372.4     -0.36      -1.28       0.83
   4    550    200     90     20     657.5     652.4     370.2     364.7     -0.78      -1.48       0.13
   5    500    220    120     20     628.4     629.3     337.1     335.9      0.14      -0.35       0.70
   6    450    180    120     20     685.3     681.3     333.4     352.3     -0.58       5.66      -6.49
   7

In [21]:
# ============================================================
# WORST LOO POINTS
# ============================================================

loo_df["Max_Error_pct"] = loo_df[
    [
        "D0_Error_pct",
        "D30_Error_pct",
        "SCI_Error_pct"
    ]
].abs().max(axis=1)

worst = loo_df.sort_values(
    "Max_Error_pct",
    ascending=False
)

print("=" * 80)
print("WORST LOO CALIBRATION POINTS")
print("=" * 80)

print(
    worst[
        [
            "Run",
            "E1",
            "E2",
            "E3",
            "E4",
            "D0_Error_pct",
            "D30_Error_pct",
            "SCI_Error_pct",
            "Max_Error_pct"
        ]
    ].head(5).to_string(index=False)
)

WORST LOO CALIBRATION POINTS
 Run    E1    E2    E3   E4  D0_Error_pct  D30_Error_pct  SCI_Error_pct  Max_Error_pct
   6 450.0 180.0 120.0 20.0     -0.581744       5.659182      -6.494574       6.494574
   0 840.0 195.0  95.0 20.0      1.839675       3.495863      -0.805370       3.495863
   1 700.0 160.0  90.0 20.0      0.346134      -0.976992       2.226649       2.226649
   4 550.0 200.0  90.0 20.0     -0.780284      -1.483168       0.125416       1.483168
   3 580.0 180.0  90.0 20.0     -0.364903      -1.284096       0.834403       1.284096


In [22]:
# ============================================================
# TEST CASE
# ============================================================

TEST_E = np.array([
    450.0,
    200.0,
    130.0,
    20.0
])

ACTUAL_TEST_D0 = 657.70
ACTUAL_TEST_D30 = 337.20

ACTUAL_TEST_SCI = (
    ACTUAL_TEST_D0 -
    ACTUAL_TEST_D30
)

print("=" * 80)
print("TEST CASE")
print("=" * 80)

print()
print("Test moduli:")
print("E1 =", TEST_E[0], "MPa")
print("E2 =", TEST_E[1], "MPa")
print("E3 =", TEST_E[2], "MPa")
print("E4 =", TEST_E[3], "MPa")

# Raw MLET
test_raw_d0, test_raw_d30 = raw_mlet_displacements(TEST_E)

print()
print("RAW PYTHON MLET")
print(f"D0  = {test_raw_d0:.2f} um")
print(f"D30 = {test_raw_d30:.2f} um")

# Features
test_row = pd.DataFrame([{
    "Raw_D0": test_raw_d0,
    "Raw_D30": test_raw_d30,
    "E1": TEST_E[0],
    "E2": TEST_E[1],
    "E3": TEST_E[2],
    "E4": TEST_E[3]
}])

X_test = make_features(test_row)

# V2.3 prediction
test_pred_d0 = np.exp(
    model_D0.predict(X_test)
)[0]

test_pred_d30 = np.exp(
    model_D30.predict(X_test)
)[0]

test_pred_sci = (
    test_pred_d0 -
    test_pred_d30
)

print()
print("V2.3 PREDICTION")
print(f"D0     = {test_pred_d0:.2f} um")
print(f"D30    = {test_pred_d30:.2f} um")
print(f"SCI300 = {test_pred_sci:.2f} um")

# Actual ERAPave
print()
print("ACTUAL ERAPAVE RESULT")
print(f"D0     = {ACTUAL_TEST_D0:.2f} um")
print(f"D30    = {ACTUAL_TEST_D30:.2f} um")
print(f"SCI300 = {ACTUAL_TEST_SCI:.2f} um")

# Errors
test_d0_error = (
    100 *
    (test_pred_d0 - ACTUAL_TEST_D0)
    / ACTUAL_TEST_D0
)

test_d30_error = (
    100 *
    (test_pred_d30 - ACTUAL_TEST_D30)
    / ACTUAL_TEST_D30
)

test_sci_error = (
    100 *
    (test_pred_sci - ACTUAL_TEST_SCI)
    / ACTUAL_TEST_SCI
)

print()
print("TEST ERRORS")
print(f"D0     = {test_d0_error:+.2f}%")
print(f"D30    = {test_d30_error:+.2f}%")
print(f"SCI300 = {test_sci_error:+.2f}%")

TEST CASE

Test moduli:
E1 = 450.0 MPa
E2 = 200.0 MPa
E3 = 130.0 MPa
E4 = 20.0 MPa

RAW PYTHON MLET
D0  = 1112.41 um
D30 = 677.93 um

V2.3 PREDICTION
D0     = 656.28 um
D30    = 335.05 um
SCI300 = 321.23 um

ACTUAL ERAPAVE RESULT
D0     = 657.70 um
D30    = 337.20 um
SCI300 = 320.50 um

TEST ERRORS
D0     = -0.22%
D30    = -0.64%
SCI300 = +0.23%


In [23]:
# ============================================================
# PREDICT NEW MODULUS COMBINATION
# ============================================================

NEW_E = np.array([
    450.0,
    180.0,
    150.0,
    20.0
])

new_raw_d0, new_raw_d30 = raw_mlet_displacements(
    NEW_E
)

new_row = pd.DataFrame([{
    "Raw_D0": new_raw_d0,
    "Raw_D30": new_raw_d30,
    "E1": NEW_E[0],
    "E2": NEW_E[1],
    "E3": NEW_E[2],
    "E4": NEW_E[3]
}])

new_X = make_features(new_row)

new_pred_d0 = np.exp(
    model_D0.predict(new_X)
)[0]

new_pred_d30 = np.exp(
    model_D30.predict(new_X)
)[0]

new_pred_sci = (
    new_pred_d0 -
    new_pred_d30
)

print("=" * 80)
print("NEW MODULUS PREDICTION")
print("=" * 80)

print()
print("E =", NEW_E)

print()
print("RAW MLET")
print(f"D0     = {new_raw_d0:.2f} um")
print(f"D30    = {new_raw_d30:.2f} um")

print()
print("V2.3 PREDICTION")
print(f"D0     = {new_pred_d0:.2f} um")
print(f"D30    = {new_pred_d30:.2f} um")
print(f"SCI300 = {new_pred_sci:.2f} um")

NEW MODULUS PREDICTION

E = [450. 180. 150.  20.]

RAW MLET
D0     = 1119.44 um
D30    = 667.00 um

V2.3 PREDICTION
D0     = 663.83 um
D30    = 329.61 um
SCI300 = 334.22 um


In [24]:
# ============================================================
# ROW 23 TARGET
# ============================================================

TARGET_D0 = 681.945
TARGET_SCI = 270.630
TARGET_D30 = 411.315

print("=" * 80)
print("ROW 23 TARGET")
print("=" * 80)

print(f"D0     = {TARGET_D0:.3f} um")
print(f"D30    = {TARGET_D30:.3f} um")
print(f"SCI300 = {TARGET_SCI:.3f} um")

ROW 23 TARGET
D0     = 681.945 um
D30    = 411.315 um
SCI300 = 270.630 um


In [ ]:
# ============================================================
# V2.3 BOUNDARY-EXPANSION FAST MODULUS SEARCH
#
# Purpose:
#   Expand the search beyond the previous restricted region
#   because the previous optimum was pushed toward:
#
#       E1 ↑
#       E2 ↓
#       E3 ↓
#
# Target Row 23:
#       D0     = 681.945 um
#       D30    = 411.315 um
#       SCI300 = 270.630 um
#
# IMPORTANT:
# This is still a SURROGATE search.
# Candidates must be verified with actual ERAPave.
# ============================================================


print("=" * 90)
print("V2.3 BOUNDARY-EXPANSION FAST MODULUS SEARCH")
print("=" * 90)


# ============================================================
# 1. ROW 23 TARGETS
# ============================================================

TARGET_D0 = 2348.045
TARGET_SCI = 654.340
TARGET_D30 = TARGET_D0 - TARGET_SCI


print()
print("ROW 23 TARGETS")
print("-" * 50)

print(f"D0     = {TARGET_D0:.3f} um")
print(f"D30    = {TARGET_D30:.3f} um")
print(f"SCI300 = {TARGET_SCI:.3f} um")


# ============================================================
# 2. EXPANDED SEARCH REGION
#
# Previous search:
#
# E1 = 450–850
# E2 = 160–250
# E3 = 90–160
# E4 = 20
#
# The best solution was pushed toward:
#
# E1 higher
# E2 lower
# E3 lower
#
# Therefore we expand in those directions.
# ============================================================


E1_values = np.arange(500, 751, 25)   # 500 ... 750
E2_values = np.arange(120, 201, 10)   # 120 ... 200
E3_values = np.arange(80, 121, 10)    # 80 ... 120
E4_values = [20]


# ============================================================
# 3. CREATE CANDIDATE GRID
# ============================================================

candidate_E = np.array([

    [
        e1,
        e2,
        e3,
        e4
    ]

    for e1 in E1_values

    for e2 in E2_values

    for e3 in E3_values

    for e4 in E4_values

])


print()
print("SEARCH REGION")
print("-" * 50)

print(
    f"E1: {E1_values[0]}–{E1_values[-1]} MPa"
)

print(
    f"E2: {E2_values[0]}–{E2_values[-1]} MPa"
)

print(
    f"E3: {E3_values[0]}–{E3_values[-1]} MPa"
)

print(
    f"E4: {E4_values}"
)

print()

print(
    "Number of candidate combinations =",
    len(candidate_E)
)


# ============================================================
# 4. FAST V2.3 PREDICTION
# ============================================================

print()
print("=" * 90)
print("RUNNING FAST SURROGATE SEARCH")
print("=" * 90)


pred_d0, pred_d30, pred_sci = (
    v23_fast_predict(
        candidate_E
    )
)


print()
print("Prediction completed.")


# ============================================================
# 5. CALCULATE ERRORS
# ============================================================

d0_error = (

    100.0
    *
    (pred_d0 - TARGET_D0)
    /
    TARGET_D0

)


d30_error = (

    100.0
    *
    (pred_d30 - TARGET_D30)
    /
    TARGET_D30

)


sci_error = (

    100.0
    *
    (pred_sci - TARGET_SCI)
    /
    TARGET_SCI

)


# ============================================================
# 6. OBJECTIVE
#
# Primary:
#       D0 + SCI
#
# Secondary:
#       D30
#
# We use a weighted score so that D0/SCl remain the main
# calibration targets while D30 helps distinguish solutions.
# ============================================================

score = (

    np.abs(d0_error)

    +

    np.abs(sci_error)

    +

    0.50 *
    np.abs(d30_error)

)


# ============================================================
# 7. RESULTS DATAFRAME
# ============================================================

search_df_expanded = pd.DataFrame({

    "E1":
        candidate_E[:, 0],

    "E2":
        candidate_E[:, 1],

    "E3":
        candidate_E[:, 2],

    "E4":
        candidate_E[:, 3],

    "Pred_D0":
        pred_d0,

    "Pred_D30":
        pred_d30,

    "Pred_SCI300":
        pred_sci,

    "D0_Error_pct":
        d0_error,

    "D30_Error_pct":
        d30_error,

    "SCI_Error_pct":
        sci_error,

    "Score":
        score

})


# ============================================================
# 8. SORT
# ============================================================

search_df_expanded = (

    search_df_expanded

    .sort_values(
        "Score",
        ascending=True
    )

    .reset_index(
        drop=True
    )

)


# ============================================================
# 9. TOP 30 CANDIDATES
# ============================================================

print()
print("=" * 125)
print("TOP 30 V2.3 BOUNDARY-EXPANSION CANDIDATES")
print("=" * 125)


print(

    search_df_expanded
    .head(30)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )

)


# ============================================================
# 10. FIND D0 + SCI <= 2%
# ============================================================

good_candidates_expanded = (

    search_df_expanded[

        (
            search_df_expanded[
                "D0_Error_pct"
            ].abs()
            <= 2.0
        )

        &

        (
            search_df_expanded[
                "SCI_Error_pct"
            ].abs()
            <= 2.0
        )

    ]

    .copy()

)


print()
print("=" * 110)
print("D0 AND SCI300 BOTH WITHIN ±2%")
print("=" * 110)


if len(good_candidates_expanded) == 0:

    print()
    print(
        "No candidate satisfies "
        "D0 <= 2% and SCI300 <= 2%."
    )

else:

    print()
    print(
        "Number of candidates =",
        len(good_candidates_expanded)
    )

    print()

    print(

        good_candidates_expanded
        .head(30)
        .to_string(
            index=False,
            float_format=lambda x: f"{x:.2f}"
        )

    )


# ============================================================
# 11. FIND ALL THREE WITHIN ±2%
# ============================================================

very_good_candidates_expanded = (

    search_df_expanded[

        (
            search_df_expanded[
                "D0_Error_pct"
            ].abs()
            <= 2.0
        )

        &

        (
            search_df_expanded[
                "D30_Error_pct"
            ].abs()
            <= 2.0
        )

        &

        (
            search_df_expanded[
                "SCI_Error_pct"
            ].abs()
            <= 2.0
        )

    ]

    .copy()

)


print()
print("=" * 110)
print("D0 + D30 + SCI300 ALL WITHIN ±2%")
print("=" * 110)


if len(very_good_candidates_expanded) == 0:

    print()
    print(
        "No candidate satisfies all three "
        "±2% criteria."
    )

else:

    print()
    print(
        "Number of candidates =",
        len(very_good_candidates_expanded)
    )

    print()

    print(

        very_good_candidates_expanded
        .head(30)
        .to_string(
            index=False,
            float_format=lambda x: f"{x:.2f}"
        )

    )


# ============================================================
# 12. BEST CANDIDATE
# ============================================================

best_expanded = (
    search_df_expanded.iloc[0]
)


print()
print("=" * 90)
print("BEST EXPANDED-SEARCH CANDIDATE")
print("=" * 90)


print()

print(
    f"E1 = {best_expanded.E1:.0f} MPa"
)

print(
    f"E2 = {best_expanded.E2:.0f} MPa"
)

print(
    f"E3 = {best_expanded.E3:.0f} MPa"
)

print(
    f"E4 = {best_expanded.E4:.0f} MPa"
)


print()

print(
    f"Predicted D0     = "
    f"{best_expanded.Pred_D0:.2f} um"
)

print(
    f"Predicted D30    = "
    f"{best_expanded.Pred_D30:.2f} um"
)

print(
    f"Predicted SCI300 = "
    f"{best_expanded.Pred_SCI300:.2f} um"
)


print()

print(
    f"D0 error     = "
    f"{best_expanded.D0_Error_pct:+.2f}%"
)

print(
    f"D30 error    = "
    f"{best_expanded.D30_Error_pct:+.2f}%"
)

print(
    f"SCI300 error = "
    f"{best_expanded.SCI_Error_pct:+.2f}%"
)

print()

print(
    f"Score = "
    f"{best_expanded.Score:.2f}"
)


# ============================================================
# 13. SHOW TOP 10 IN A CLEAN FORMAT
# ============================================================

print()
print("=" * 120)
print("TOP 10 CANDIDATES — CLEAN VIEW")
print("=" * 120)


top10 = search_df_expanded.head(10).copy()


print(
    top10[
        [
            "E1",
            "E2",
            "E3",
            "E4",
            "Pred_D0",
            "Pred_D30",
            "Pred_SCI300",
            "D0_Error_pct",
            "D30_Error_pct",
            "SCI_Error_pct",
            "Score"
        ]
    ]

    .to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)


# ============================================================
# 14. SAVE RESULTS
# ============================================================

OUTPUT_FILE = (
    "V2_3_Row23_Boundary_Expansion_Search.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    search_df_expanded.to_excel(
        writer,
        sheet_name="All_Candidates",
        index=False
    )

    good_candidates_expanded.to_excel(
        writer,
        sheet_name="D0_SCI_Under_2pct",
        index=False
    )

    very_good_candidates_expanded.to_excel(
        writer,
        sheet_name="All_Under_2pct",
        index=False
    )


print()
print("=" * 90)
print("SEARCH COMPLETE")
print("=" * 90)

print()

print(
    "Results saved to:",
    OUTPUT_FILE
)

print()

print(
    "IMPORTANT:"
)

print(
    "The best candidate is NOT yet a final ERAPave label."
)

print(
    "Verify the best candidates using actual ERAPave."
)

V2.3 BOUNDARY-EXPANSION FAST MODULUS SEARCH

ROW 23 TARGETS
--------------------------------------------------
D0     = 681.945 um
D30    = 411.315 um
SCI300 = 270.630 um

SEARCH REGION
--------------------------------------------------
E1: 500–750 MPa
E2: 120–200 MPa
E3: 80–120 MPa
E4: [20]

Number of candidate combinations = 495

RUNNING FAST SURROGATE SEARCH

Prediction completed.

TOP 30 V2.3 BOUNDARY-EXPANSION CANDIDATES
 E1  E2  E3  E4  Pred_D0  Pred_D30  Pred_SCI300  D0_Error_pct  D30_Error_pct  SCI_Error_pct  Score
725 130  80  20   660.04    389.66       270.38         -3.21          -5.26          -0.09   5.94
725 120  80  20   664.11    390.89       273.22         -2.62          -4.97           0.96   6.05
700 140  80  20   661.05    388.39       272.65         -3.06          -5.57           0.75   6.60
700 150  80  20   657.56    387.34       270.22         -3.58          -5.83          -0.15   6.64
750 120  80  20   659.17    390.89       268.28         -3.34          -4.9

In [29]:
# ============================================================
# BEST V2.3 CANDIDATE
# ============================================================

best = search_df.iloc[0]

print("=" * 80)
print("BEST V2.3 CANDIDATE")
print("=" * 80)

print()
print(
    f"E1 = {best.E1:.0f} MPa"
)

print(
    f"E2 = {best.E2:.0f} MPa"
)

print(
    f"E3 = {best.E3:.0f} MPa"
)

print(
    f"E4 = {best.E4:.0f} MPa"
)

print()
print(
    f"Predicted D0     = {best.Pred_D0:.2f} um"
)

print(
    f"Predicted D30    = {best.Pred_D30:.2f} um"
)

print(
    f"Predicted SCI300 = {best.Pred_SCI300:.2f} um"
)

print()
print(
    f"D0 error     = {best.D0_Error_pct:+.2f}%"
)

print(
    f"D30 error    = {best.D30_Error_pct:+.2f}%"
)

print(
    f"SCI300 error = {best.SCI_Error_pct:+.2f}%"
)

BEST V2.3 CANDIDATE

E1 = 700 MPa
E2 = 140 MPa
E3 = 80 MPa
E4 = 30 MPa

Predicted D0     = 661.05 um
Predicted D30    = 388.39 um
Predicted SCI300 = 272.65 um

D0 error     = -3.06%
D30 error    = -5.57%
SCI300 error = +0.75%


In [30]:
# ============================================================
# V2.3 DECISION
# ============================================================

print("=" * 80)
print("V2.3 DECISION")
print("=" * 80)

if (
    loo_d0_mape <= 2.0
    and
    loo_d30_mape <= 2.0
    and
    loo_sci_mape <= 2.0
):

    print()
    print("V2.3 PASSES the 2% local LOO criterion.")

else:

    print()
    print("V2.3 DOES NOT PASS the 2% local LOO criterion.")

print()
print("IMPORTANT:")
print(
    "This validation uses only one pavement structure "
    "(Row 23)."
)

print(
    "Therefore it is NOT sufficient to predict the "
    "15,000-row dataset."
)

V2.3 DECISION


NameError: name 'loo_d0_mape' is not defined

In [ ]:
# ============================================================
# SAVE RESULTS
# ============================================================

OUTPUT_FILE = (
    "ERAPAVE_V2_3_Row23_Calibration.xlsx"
)

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    df.to_excel(
        writer,
        sheet_name="Training",
        index=False
    )

    loo_df.to_excel(
        writer,
        sheet_name="LOO",
        index=False
    )

    search_df.head(100).to_excel(
        writer,
        sheet_name="Best_Candidates",
        index=False
    )

print()
print("Saved:")
print(OUTPUT_FILE)